In [ ]:
!pip install kaggle

In [ ]:
from google.colab import files
files.upload()

Saving kaggle.json to kaggle.json


{'kaggle.json': b'{"username":"mohamedkharrat122","key":"499ba8e6a5356890a01d9837a714c755"}'}

In [ ]:
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

In [ ]:
!kaggle datasets download -d paultimothymooney/chest-xray-pneumonia

Dataset URL: https://www.kaggle.com/datasets/paultimothymooney/chest-xray-pneumonia
License(s): other
100% 2.29G/2.29G [00:57<00:00, 42.7MB/s]



In [ ]:
!unzip -q chest-xray-pneumonia.zip

In [ ]:
import os
print(os.listdir("chest_xray"))
# should print: ['train', 'test', 'val']

['train', 'chest_xray', '__MACOSX', 'test', 'val']


In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from sklearn.utils.class_weight import compute_class_weight
import numpy as np

train_dir = "chest_xray/train"
val_dir   = "chest_xray/val"
test_dir  = "chest_xray/test"

In [ ]:
train_datagen = ImageDataGenerator(
    rescale=1./255,
    horizontal_flip=True,
    zoom_range=0.1,
    rotation_range=10,
    width_shift_range=0.1,
    height_shift_range=0.1
)
test_datagen = ImageDataGenerator(rescale=1./255)

train_gen = train_datagen.flow_from_directory(
    train_dir, target_size=(224,224),
    batch_size=32, class_mode="binary"
)
val_gen = test_datagen.flow_from_directory(
    test_dir, target_size=(224,224),
    batch_size=32, class_mode="binary"
)
print("Classes:", train_gen.class_indices)

Found 5216 images belonging to 2 classes.
Found 624 images belonging to 2 classes.
Classes: {'NORMAL': 0, 'PNEUMONIA': 1}


In [ ]:
weights = compute_class_weight(
    class_weight="balanced",
    classes=np.array([0,1]),
    y=train_gen.classes
)
class_weights = dict(enumerate(weights))
print("Class weights:", class_weights)

Class weights: {0: np.float64(1.9448173005219984), 1: np.float64(0.6730322580645162)}


In [ ]:
base_model = ResNet50(
    weights="imagenet",
    include_top=False,
    input_shape=(224,224,3)
)
base_model.trainable = False

model = models.Sequential([
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.BatchNormalization(),
    layers.Dense(256, activation="relu"),
    layers.Dropout(0.5),
    layers.Dense(128, activation="relu"),
    layers.Dropout(0.3),
    layers.Dense(1, activation="sigmoid")
])

94765736/94765736 ━━━━━━━━━━━━━━━━━━━━ 3s 0us/step


In [ ]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-3),
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

early_stop = tf.keras.callbacks.EarlyStopping(
    monitor="val_loss",
    patience=3,
    restore_best_weights=True
)

history1 = model.fit(
    train_gen,
    epochs=10,
    validation_data=val_gen,
    class_weight=class_weights,
    callbacks=[early_stop]
)

Epoch 1/10
163/163 ━━━━━━━━━━━━━━━━━━━━ 145s 780ms/step - accuracy: 0.8445 - loss: 0.3603 - val_accuracy: 0.7692 - val_loss: 0.6156
Epoch 2/10
163/163 ━━━━━━━━━━━━━━━━━━━━ 115s 705ms/step - accuracy: 0.8821 - loss: 0.2750 - val_accuracy: 0.7228 - val_loss: 0.5515
Epoch 3/10
163/163 ━━━━━━━━━━━━━━━━━━━━ 114s 702ms/step - accuracy: 0.8834 - loss: 0.2673 - val_accuracy: 0.7484 - val_loss: 0.4530
Epoch 4/10
163/163 ━━━━━━━━━━━━━━━━━━━━ 115s 703ms/step - accuracy: 0.8990 - loss: 0.2334 - val_accuracy: 0.8349 - val_loss: 0.3626
Epoch 5/10
163/163 ━━━━━━━━━━━━━━━━━━━━ 113s 691ms/step - accuracy: 0.9059 - loss: 0.2161 - val_accuracy: 0.8734 - val_loss: 0.3627
Epoch 6/10
163/163 ━━━━━━━━━━━━━━━━━━━━ 113s 695ms/step - accuracy: 0.8997 - loss: 0.2260 - val_accuracy: 0.8814 - val_loss: 0.3264
Epoch 7/10
163/163 ━━━━━━━━━━━━━━━━━━━━ 112s 685ms/step - accuracy: 0.9053 - loss: 0.2261 - val_accuracy: 0.8638 - val_loss: 0.3368
Epoch 8/10
163/163 ━━━━━━━━━━━━━━━━━━━━ 112s 689ms/step - accuracy: 0.9084 -

In [ ]:
loss, acc = model.evaluate(val_gen)
print(f"\nTest accuracy: {acc*100:.2f}%")

20/20 ━━━━━━━━━━━━━━━━━━━━ 6s 280ms/step - accuracy: 0.8814 - loss: 0.3264

Test accuracy: 88.14%


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# save to Google Drive (safe forever)
model.save("/content/drive/MyDrive/pneumonia_model.keras")
print("Saved! ✅")

Mounted at /content/drive
Saved! ✅
